In [ ]:
import os, sys
_r = os.path.abspath(os.getcwd())
while _r != "/" and not os.path.exists(os.path.join(_r, "perm_paths.py")):
    _r = os.path.dirname(_r)
sys.path.insert(0, _r)
import perm_paths as P

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, P.FIGURES)
import figures as F
import numpy as np

# ── THE cutoff for this notebook. Set it HERE, once. Every panel below reads
# F.TOP_N, and every file lands in panels/top<N>/ with its svg alongside.
# It is stored in the environment, so %autoreload cannot reset it mid-session.
F.use_top_n(300)          # <- the only place to change it
F.LABEL_MODE = "celltype" 

# global knobs, if you want to flip them for the whole figure
# F.RESULTS_TREE = "python_implements"     # or "data" for the authors' originals
# F.PUB_RC["font.size"] = 8
print("datasets:", F.active_datasets(), "| excluded:", sorted(F.DATASET_EXCLUDE))
print("methods :", list(F.METHOD_PREFIX))

# shared margins: same figsize + grid + margins -> identical panel boxes in Fig 1b and 1d
MARGINS_2COL = dict(left=.10, right=.99, bottom=.11, top=.92, wspace=.12, hspace=.62)
MARGINS_ROW4 = dict(left=.065, right=.995, bottom=.21, top=.82, wspace=.14)


In [ ]:
# === Source Data export helpers — run before the panel cells ===
import os, glob, numpy as np, pandas as pd

SD_ROOT = P.OUTPUTS + "/source_data"
SD_FIG  = "figure1"                      # <-- "figure1" in figure1.ipynb
SD_DIR  = f"{SD_ROOT}/{SD_FIG}"
os.makedirs(SD_DIR, exist_ok=True)


def _flat(lab):
    return str(lab).replace("\n", " ").strip()

def sd_violin(res, panel, stats=None, mem="memory", bg="background", x="group"):
    """Every PLOTTED point of a violin panel, one row per value."""
    rows = [pd.DataFrame({x: _flat(lab), "set": gname,
                          "value": np.asarray(v[key], float).ravel()})
            for lab, v in res.items()
            for key, gname in (("scg_values", mem), ("non_scg_values", bg))]
    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f"{SD_DIR}/{panel}_values.csv", index=False)
    print(f"  source data -> {panel}_values.csv  ({len(df):,} rows)")
    if stats is not None:
        pd.DataFrame(stats).to_csv(f"{SD_DIR}/{panel}_stats.csv", index=False)
        print(f"  source data -> {panel}_stats.csv")
    return df

def sd_table(obj, panel, index=True):
    """A panel whose plotted content is already tabular."""
    df = pd.DataFrame(obj)
    df.to_csv(f"{SD_DIR}/{panel}_values.csv", index=index)
    print(f"  source data -> {panel}_values.csv  ({len(df):,} rows)")
    return df


In [ ]:
"""
purple/orange
COL = {"Michaels": "#9d4edd",     # human
       "Harmange": "#b67be6",     # human
       "Bandler":  "#fa8c02",     # mouse
       "Weinreb":  "#ffad03"}     # mouse
SPECIES_COL = {"human": "#72369d", "mouse": "#f56a00"}


"""

In [ ]:
# === Fig 1b — sunburst plot ===

RINGS = 4          # 4 = species / tissue / dataset / group,   2 = dataset / group
BURST = .30        # burst length (fraction of rim radius); 0 = no bursts

# ── colours: edit here ─────────────────────────────────────────────────────
COL = {"Michaels": "#9d4edd",     # human
       "Harmange": "#e0aaff",     # human
       "Bandler":  "#48bfe3",     # mouse
       "Weinreb":  "#5aedc9"}     # mouse
SPECIES_COL = {"human": "#72369d", "mouse": "#64dfdf"}
F.DATASET_COLORS.update({F.dkey(k): v for k, v in COL.items()})   # author name -> internal key


ORDER = ["T-lineage", "Harmange", "Bandler", "LARRY"]     # Weinreb (LARRY) last
SYS  = {"Michaels": "T cells", "Harmange": "Melanoma", "Bandler": "Brain", "Weinreb": "HSPCs",
        "Mold": "Brain"}
SPEC = {"Michaels": "human", "Harmange": "human", "Bandler": "mouse", "Weinreb": "mouse",
        "Mold": "mouse"}
GL = {("Weinreb", "2.0"): "T1", ("Weinreb", "4.0"): "T2", ("Weinreb", "6.0"): "T3",
      ("Harmange", "E"): "Enriched", ("Harmange", "M"): "Mixed",
      ("Mold", "brain1"): "b1", ("Mold", "brain2"): "b2"}


# wedge width = the group's share of clones (>=2 cells); every dataset takes 1/4 of the circle
rows, w, counts, I = F.lineage_design_rows(
    datasets={k: F.LINEAGE_DATASETS[k] for k in ORDER},
    systems=SYS, group_labels=GL, weight="clones_ge2", hub="cells_in_ge2",
    share="dataset")
full = [(SPEC[F.alabel(r[2])], r[1], F.alabel(r[2]), r[3]) for r in rows]
assert [r[3] for r in full] == list(I.group)              # rows and counts line up

# human|mouse boundary vertical, humans on the right
cum   = np.concatenate([[0], np.cumsum(w)]) / sum(w) * 360
start = cum[next(i for i, r in enumerate(full) if r[0] == "mouse")] - 90

if RINGS == 4:
    LR, ring_kw = full, dict(family_level=2, count_level=3, fs=7, r0=.26, ring_w=.27, gap=.012,
                             level_colors={0: SPECIES_COL})
else:
    LR, ring_kw = [(d, g) for s, t, d, g in full], dict(family_level=0, count_level=1, fs=8,
                                                         r0=.36, ring_w=.50, gap=.016, level_colors={})

# burst length = cells in clones >=2, keyed by the FULL row so Michaels T1 and Weinreb T1 stay apart
CELLS = {tuple(r): int(n) for r, n in zip(LR, I.cells_in_ge2)}
r_rim = ring_kw["r0"] + RINGS * ring_kw["ring_w"]

fig, ax, nd = F.fig_design_sunburst(
    rows=LR, weights=w, start_angle=start,
    colors=[F.DATASET_COLORS[k] for k in ORDER],    # each dataset keeps its colour
    ring_labels=None, label_gap_deg=0,             # no ring names, no notch
    #ring_note="T, timepoint; E, embryonic day",
    label_orient="tangential", curved_labels=True, min_span_inside=8,
    counts=CELLS, hub_bars=BURST > 0, count_bars="extend",
    hub_scale="log", ray_span=BURST, ray_min_frac=.2,   # LOG scale -> say so in the legend
    edge_labels=None,                              # no rim numbers
    lighten_step=.12, edge_lw=1.2, lim=r_rim + BURST + .04,
    figsize=(2.6, 2.6),                            # drawn at print size: place at 100%
    save=f"{F.OUTDIR}/fig1b_design_sunburst.png", dpi=400, **ring_kw)

sd_table(pd.DataFrame({"species":      [r[0] for r in full],
                       "system":       [r[1] for r in full],
                       "dataset":      [r[2] for r in full],
                       "group":        [r[3] for r in full],
                       "wedge_weight": list(w),
                       "clones":       list(I.clones),
                       "clones_ge2":   list(I.clones_ge2),
                       "cells_in_ge2": list(I.cells_in_ge2)}),
         "Fig1b", index=False)


In [ ]:
# === Fig 1c — amount of identified memory genes across methods ===

fig, agree = F.fig_agreement(
    "all", top_n=F.TOP_N, logy=False, fs=10, ncol=4,
    panel_w=3.6, panel_h=2.5, bar_width=.95,
    colors=[F.C_OBS, "white", "white"],                      # white bars
    hatches=("", "\\\\\\\\", "xxx"),         # diagonal | grid | vertical, per group
    hatch_color=F.C_OBS, hatch_lw=2.0,      # red texture
    hatch_edge_lw=.8,                      # red outline, so a sparse bar still has an edge
    annot_color=F.C_OBS,                   # counts in red, not in the (white) bar colour
    yticklabels=False, ylabel="number of genes",
    #yticks=range(0, 600, 100),
    legend_handlelength=2.0,    # width of the key, default 1.0
    legend_handleheight=1.2,    # height of the key, matplotlib default 0.7
    threshold_ticks="all", panel_letters=None,
    annotate="observed", annot_fontsize=9,
    enrich=None, null_bars="group",
    label_celltype=True,
    margins=MARGINS_2COL,
    save=f"{F.OUTDIR}/fig1c_method_agreement_line.png")

agree[["dataset", "group_label", "threshold", "K", "observed", "null_mean", "p"]].round(3)
sd_table(agree, "Fig1c", index=False)

In [ ]:
# === Fig 1d — memory genes across all datasets ===

fig_c, res_c = F.fig_cross_dataset(
    panel="bars",                 # left half only; the matrix is in the supplement
    min_methods=2, top_n=F.TOP_N, pooled_depth=True, thresholds=(2, 3, 4),
    null_mode="matched", n_perm=1000, seed=0, logy=False,
    fs=13, figsize=(4.6, 3.9),
    z_metric=None,null_annot_fmt="{:.0f}",
    threshold_below=True, yticklabels=False,
    save=f"{F.OUTDIR}/fig1d_cross_dataset.png", dpi=400,
)
res_c["stats"]
sd_table(res_c["stats"], "Fig1d", index=False)

In [ ]:
# === Fig 1e — expression: genes called by >=2 methods vs a same-size background ===

char = F.characterisation_table(top_n=F.TOP_N, tiers=(2,))   # background = ALL never-called genes

res_d = {}
for ds in F.active_datasets():                     # Michaels, Harmange, Weinreb, Bandler
    d = char[char.dataset == ds]
    res_d[F.dlabel(ds)] = dict(
        scg_values=d.loc[d.group == "≥2", "expression"].to_numpy(float),
        non_scg_values=d.loc[d.group == "background", "expression"].to_numpy(float))

fig_d, ax_d, stats_d = F.fig_responsiveness_violin(
    res=res_d, mem_label="≥2 methods", bg_label="background",
    bg_color=F.C_NULL,                             # memory red (the default), background grey
    log_y=True,                                    # log10 of mean expression (genes at 0 dropped)
    annot="stars",                                 # BH-corrected Mann-Whitney across the 4 datasets
    show_medians=False, fill=False, median_inside=True,
    median_fmt="{:.2f}",                           # CHANGED: print the median inside each half
    xlabel="", ylabel="log₁₀ mean expression",
    legend=False,
    fs=10, figsize=(4.4, 2.5),
    save=f"{F.OUTDIR}/fig1e_expression_split.png", dpi=400,
)
stats_d[["method", "n_mem", "n_bg", "median_mem", "median_bg", "cliffs_d", "q"]]
sd_violin(res_d, "Fig1e", stats=stats_d, mem="≥2 methods", bg="background", x="dataset")

In [ ]:
# === Fig 1f — molecular features: half-life, bursting, promoter CpG, per group ===
mf_table = F.molecular_feature_table()
mf_results, mf_stats = F.molecular_feature_tests(mf_table)
fig_mf, ax_mf, _ = F.fig_molecular_features(
    results=mf_results, name="fig1f_molecular_features", save=True, dpi=400)
sd_table(mf_stats, "fig1f", index=False)
mf_stats.round(4)

In [ ]:
# === Fig 1g — expression autocorrelation across cell divisions ===
fig_ac, axes_ac, ac_data = F.fig_expression_autocorrelation(
    name="fig1g_expression_autocorrelation", save=True, dpi=400)